# 707. Design Linked List

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** linked-list, design
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-linked-list/)

Design your own linked list. Singly or doubly - your choice. A node in a singly
linked list has `val` and `next`; a doubly linked list adds `prev`. All nodes are
**0-indexed**.

Implement the `MyLinkedList` class:

- `MyLinkedList()` initializes the object.
- `get(index)` returns the value of the `index`-th node, or `-1` if the index is
  invalid.
- `addAtHead(val)` inserts a node before the first element.
- `addAtTail(val)` appends a node at the end.
- `addAtIndex(index, val)` inserts a node **before** the `index`-th node. If
  `index` equals the length, the node is **appended**. If `index` is greater than
  the length, the node is **not inserted**.
- `deleteAtIndex(index)` deletes the `index`-th node, if the index is valid.

---

### Example 1

```
Input:  ["MyLinkedList","addAtHead","addAtTail","addAtIndex","get","deleteAtIndex","get"]
        [[],            [1],        [3],        [1,2],       [1],  [1],            [1]]
Output: [null,          null,       null,       null,        2,    null,           3]

MyLinkedList myLinkedList = new MyLinkedList();
myLinkedList.addAtHead(1);
myLinkedList.addAtTail(3);
myLinkedList.addAtIndex(1, 2);   // list is now  1 -> 2 -> 3
myLinkedList.get(1);             // returns 2
myLinkedList.deleteAtIndex(1);   // list is now  1 -> 3
myLinkedList.get(1);             // returns 3
```

---

### Constraints

- `0 <= index, val <= 1000`
- Do not use the built-in linked list library.
- At most `2000` calls will be made to `get`, `addAtHead`, `addAtTail`,
  `addAtIndex` and `deleteAtIndex`.

You have been building pieces of this class for two weeks: the queue with
`first`/`last` in #104, the `addFirst`-in-`O(1)` list in #103, the stack in #155,
the ring of nodes in #622. This is all of it at once, with an **index** API - and
the index is where the problem hides.


## Before you write anything

Nothing here is a new algorithm. Every line is pointer moving you have already
done. What is new is a **contract**: five methods, three of which take an index,
and each one draws the line between "valid" and "invalid" in a *different* place.
Get that table wrong and no amount of correct pointer surgery saves you.

**1.** Fill this in from the statement, for a list of length `size`:

```
              smallest legal index   largest legal index   what happens outside
get                    ?                     ?                    ?
addAtIndex             ?                     ?                    ?
deleteAtIndex          ?                     ?                    ?
```

One of the three accepts `size` itself and the other two do not. Say in one
sentence why that is not an inconsistency but the only sensible reading of
"insert **before** index `i`".

**2.** To insert before position `i`, you need to hold the node at position
`i - 1` - the one whose `.next` you are about to rewrite. What is that node when
`i == 0`?

There isn't one. So either `addAtHead` and `addAtIndex(0, v)` become their own
special code path with their own bugs... or you make the missing node **exist**.
What if the list permanently held one extra node at the front that carries no
data, and `head` meant "the node after it"? Answer the question again with that
node in place.

**3.** With that extra node (a **sentinel**, or dummy head), write the loop that
lands you on "the node before index `i`" - it is two lines. How many steps does it
take for `i == 0`? For `i == size`? Then check: do insert-at-`0` and
insert-at-`size` now run the *same* code, with no `if`?

**4.** Do you keep a `self.size` field, or count by walking when you need it?
Price both: per call, and in bug-per-line of the table in question 1. Then, if you
keep it: name every method that must update it, and say what an off-by-one in one
of them does to `get` three calls later.

**5.** `addAtTail` with only a head pointer is `O(n)` - you walk the whole list to
find the end. So add a `self.tail` pointer and it is `O(1)`. Now trace
`deleteAtIndex(size - 1)` on a **singly** linked list with that tail pointer:
after the last node is gone, what must `tail` point at, and can you get there from
where you are standing? Why does this one trace turn into the argument for a
**doubly** linked list?

**6.** Doubly linked, with a sentinel at **both** ends. Write the four assignments
that splice a new node between `a` and `b`. How many `is None` checks does that
routine need, and why exactly zero? What do the two sentinels cost you - in memory,
and in `size` bookkeeping?

**7.** Cost table, because this problem is the best excuse you will get to write
it down. For a Python `list` and for your class, fill in `O(?)`:

```
                       Python list      your linked list
read index i               ?                  ?
insert at 0                ?                  ?
append at the end          ?                  ?
delete at 0                ?                  ?
delete at the end          ?                  ?
memory per element         ?                  ?
```

Now answer the question the table sets up: given that, why does Python's standard
library ship **no** linked list at all - and what does `collections.deque` do that
neither a `list` nor this class does?

**8.** How do you test this? Only `get` returns anything; the other four return
`None`. So a broken link, a stale `tail` or a wrong `size` is **invisible** at the
moment you cause it, and shows up as a wrong answer several calls later. What
would you have to do after *every* mutation to make corruption visible
immediately? (That is exactly what the harness below does - guess it before you
read it.)


## Two routes - A to submit, B to keep

**A - singly linked, one sentinel, a `size` counter** *(write this first)*
`self.head = Node(0)` is a dummy that never holds data and never moves;
`self.size = 0`. Every index method starts by walking to "the node before index
`i`", which for `i == 0` is the sentinel itself - so there is no head special
case anywhere in the class. `addAtHead(v)` is `addAtIndex(0, v)`, `addAtTail(v)`
is `addAtIndex(self.size, v)`, and you write the splicing **once**.

Costs: `get` and both index methods `O(i)`, `addAtHead` `O(1)`,
`addAtTail` `O(n)`, memory `O(n)`. That is accepted, and the shape to submit.

**B - doubly linked, sentinel at both ends** *(the version worth keeping)*
`self.head` and `self.tail` are both dummies, wired to each other at
construction: `head.next = tail`, `tail.prev = head`. Now the list is *never*
empty from the code's point of view, insertion is the same four assignments in
every case, and deletion is two - `node.prev.next = node.next` and
`node.next.prev = node.prev` - with no `if` in sight. `addAtTail` becomes `O(1)`
because `tail.prev` is the last real node, and `get(i)` can walk from whichever
end is closer: `O(min(i, n - i))`.

The price: one extra pointer per node, two dummy nodes, and **two** links to fix
in every mutation instead of one. Forget one direction and the list is fine
forwards and corrupt backwards - which the harness catches only because it reads
the whole list back after every single call.

> **Sentinels are the whole lesson.** One node in `__init__` that holds no data
> deletes an `if` from four methods. You have met this move before under other
> names: #155 gave every node its own running minimum so `pop` needed no
> recomputation, #355 stamped one clock on every tweet so ordering was free, #622
> kept one `count` so `isEmpty` and `isFull` stopped being the same test. Same
> instinct every time - **pay once, in state, to stop paying in special cases.**

Write A, run the tests, rewrite the class as B, run the same tests. And be honest
about one thing the harness cannot see: a `MyLinkedList` that wraps a Python
`list` passes every test below. The nodes are the assignment.


In [23]:

class Node :
    def __init__(self, val, next=None):
        self.val = val
        self.next = next

class MyLinkedList:
    def __init__(self):
        self.head:Node = None
        self.tail:Node = None
        self.length = 0

    def get(self, index: int) -> int:
        if index < 0 : index = abs(index)
        if index >= self.length : return -1
        current = self.head
        while current and index > 0:
            current = current.next
            index -= 1
        return current.val

    def addAtHead(self, val: int) -> None:
        value = Node(val)
        self.length += 1
        if self.head == None:
            self.head = value
            self.tail = value
            return
        value.next = self.head
        self.head = value
        return
    def addAtTail(self, val: int) -> None:
        value = Node(val)
        self.length += 1
        if self.tail == None:
            self.tail = value
            self.head = value
            return
        self.tail.next = value
        self.tail = value
        return
    def addAtIndex(self, index: int, val: int) -> None:
        if index <= 0:
            self.addAtHead(val)
            return
        if index > self.length:
            return
        if index == self.length:
            self.addAtTail(val)
            return
        prev = self.head
        current = self.head
        i = Node(val)
        while current and index > 0 :
            prev = current
            current = current.next
            index -=1
        prev.next = i
        i.next = current
        self.length += 1
        return

    def deleteAtIndex(self, index: int) -> None:
        if index >= self.length:
            return
        if index == 0:
            if self.head == self.tail:
                self.head = None
                self.tail = None
            else :
                self.head = self.head.next
        else :
            current = self.head
            prev = self.head
            while current and index > 0:
                prev = current
                current = current.next
                index -=1
            if self.tail == current :
                self.tail = prev
            prev.next = current.next
        self.length -= 1



### The test harness

Question 8's answer. Four of the five methods return `None`, so comparing return
values is nearly blind - a corrupted list keeps quietly agreeing with you.

So `check` replays the operations against your class **and** against a plain
Python `list` as the model, and after **every single call** it reads your whole
list back - `get(0)`, `get(1)`, ... `get(size - 1)` - and compares it to the
model. It also asks for `get(size)`, one past the end, which must be `-1`: that is
what catches a `size` that drifted out of step with the nodes.

The consequence is that a failure is reported at the operation that *caused* it,
not three calls later. Every exception is caught and reported too, so a
`AttributeError: 'NoneType' object has no attribute 'next'` shows up as a normal
FAIL line with the call that raised it.

`stress` builds long random sequences and checks them the same way. The seed makes
every run identical. Run this cell; don't edit it.


In [24]:
import random


def check(ops, args):
    """Replay (ops, args) against MyLinkedList and a Python list model.

    After every call, read the whole list back with get() and compare.
    Returns (ok, log); on failure the log ends with what broke.
    """
    ll, model, log = None, [], []

    def read_back():
        """The list as your class reports it, plus the one-past-the-end probe."""
        return [ll.get(i) for i in range(len(model))], ll.get(len(model))

    for op, a in zip(ops, args):
        call = f"{op}({', '.join(map(str, a))})"
        try:
            if op == "MyLinkedList":
                ll, model = MyLinkedList(), []
                log.append(call)
                continue

            got = getattr(ll, op)(*a)

            if op == "get":
                i = a[0]
                want = model[i] if 0 <= i < len(model) else -1
                log.append(f"{call} -> {got!r}      list {model}")
                if got != want:
                    log.append(f"   !! {call} must return {want!r}, got {got!r}")
                    return False, log
            else:
                if op == "addAtHead":
                    model.insert(0, a[0])
                elif op == "addAtTail":
                    model.append(a[0])
                elif op == "addAtIndex":
                    i, v = a
                    if i <= len(model):          # i > size: not inserted
                        model.insert(i, v)
                elif op == "deleteAtIndex":
                    i = a[0]
                    if 0 <= i < len(model):
                        model.pop(i)
                log.append(f"{call}            list {model}")
                if got is not None:
                    log.append(f"   !! {call} must return None, got {got!r}")
                    return False, log

            seen, past_end = read_back()
            if seen != model:
                log.append(f"   !! after {call} your list reads back {seen}")
                log.append(f"      it should be                    {model}")
                return False, log
            if past_end != -1:
                log.append(f"   !! get({len(model)}) is one past the end: must be -1, got {past_end!r}")
                return False, log

        except Exception as e:
            log.append(f"   !! {call} raised {type(e).__name__}: {e}")
            return False, log

    return True, log


def stress(calls, seed=0, max_index=6):
    """Random operations, checked the same way after every call."""
    random.seed(seed)
    ops, args = ["MyLinkedList"], [[]]
    for _ in range(calls):
        op = random.choice(["addAtHead", "addAtTail", "addAtIndex", "addAtIndex",
                            "deleteAtIndex", "get", "get"])
        if op in ("addAtHead", "addAtTail"):
            a = [random.randint(0, 1000)]
        elif op == "addAtIndex":
            a = [random.randint(0, max_index), random.randint(0, 1000)]
        else:
            a = [random.randint(0, max_index)]
        ops.append(op)
        args.append(a)
    return check(ops, args)


def report(name, ok, log, tail=6):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")


In [25]:
# tests
CASES = [
    ("the LeetCode example",
     ["MyLinkedList","addAtHead","addAtTail","addAtIndex","get","deleteAtIndex","get"],
     [[],            [1],        [3],        [1,2],       [1],  [1],            [1]]),

    ("every method on an empty list",
     ["MyLinkedList","get","deleteAtIndex","addAtIndex","get","addAtIndex","get"],
     [[],            [0],  [0],            [1,9],       [0],  [0,5],       [0]]),

    ("addAtIndex at exactly size appends; past size is ignored",
     ["MyLinkedList","addAtHead","addAtIndex","addAtIndex","get","get","get"],
     [[],            [1],        [1,2],       [4,99],      [0],  [1],  [2]]),

    ("question 2: addAtIndex(0, v) on a NON-empty list goes before the head",
     ["MyLinkedList","addAtTail","addAtTail","addAtIndex","get","get","get","addAtIndex","get"],
     [[],            [1],        [2],        [0,9],       [0],  [1],  [2],  [0,8],       [0]]),

    ("delete the head, the tail, and the middle",
     ["MyLinkedList","addAtTail","addAtTail","addAtTail","addAtTail",
      "deleteAtIndex","get","deleteAtIndex","get","deleteAtIndex","get"],
     [[],            [1],        [2],        [3],        [4],
      [0],            [0],  [2],            [1],  [1],            [0]]),

    ("question 5: delete the last node, then addAtTail (stale tail pointer)",
     ["MyLinkedList","addAtHead","addAtTail","deleteAtIndex","addAtTail","get","get","get"],
     [[],            [1],        [2],        [1],            [7],        [0],  [1],  [2]]),

    ("delete the only element, then build again",
     ["MyLinkedList","addAtHead","deleteAtIndex","get","addAtTail","get","addAtHead","get"],
     [[],            [5],        [0],            [0],  [8],        [0],  [4],        [0]]),

    ("ten addAtHead - the list comes out reversed",
     ["MyLinkedList"] + ["addAtHead"]*10 + ["get","get"],
     [[]] + [[i] for i in range(10)] + [[0],[9]]),

    ("drain from the front until empty",
     ["MyLinkedList","addAtTail","addAtTail","addAtTail",
      "deleteAtIndex","deleteAtIndex","deleteAtIndex","get","deleteAtIndex","addAtTail","get"],
     [[],            [1],        [2],        [3],
      [0],            [0],            [0],            [0],  [0],            [6],        [0]]),

    ("drain from the back until empty - the tail-pointer killer",
     ["MyLinkedList","addAtTail","addAtTail","addAtTail",
      "deleteAtIndex","deleteAtIndex","deleteAtIndex","get","addAtTail","get"],
     [[],            [1],        [2],        [3],
      [2],            [1],            [0],            [0],  [6],        [0]]),

    ("value 0 is legal, and so is index 0 everywhere",
     ["MyLinkedList","addAtHead","get","addAtIndex","get","get","deleteAtIndex","get"],
     [[],            [0],        [0],  [0,0],       [0],  [1],  [0],            [0]]),
]

for name, ops, args in CASES:
    report(name, *check(ops, args))

# random sequences - the last one is the constraint ceiling (2000 calls)
for calls, seed, mx in [(50,1,3), (200,2,6), (500,3,10), (2000,4,20)]:
    report(f"stress: {calls} random calls (seed {seed}, indices 0..{mx})",
           *stress(calls, seed, mx))

# see it, do not just trust the pass/fail
print("\ntrace of the LeetCode example:")
for line in check(CASES[0][1], CASES[0][2])[1]:
    print("  " + line)


OK   the LeetCode example
OK   every method on an empty list
OK   addAtIndex at exactly size appends; past size is ignored
OK   question 2: addAtIndex(0, v) on a NON-empty list goes before the head
OK   delete the head, the tail, and the middle
OK   question 5: delete the last node, then addAtTail (stale tail pointer)
OK   delete the only element, then build again
OK   ten addAtHead - the list comes out reversed
OK   drain from the front until empty
OK   drain from the back until empty - the tail-pointer killer
OK   value 0 is legal, and so is index 0 everywhere
OK   stress: 50 random calls (seed 1, indices 0..3)
OK   stress: 200 random calls (seed 2, indices 0..6)
OK   stress: 500 random calls (seed 3, indices 0..10)
OK   stress: 2000 random calls (seed 4, indices 0..20)

trace of the LeetCode example:
  MyLinkedList()
  addAtHead(1)            list [1]
  addAtTail(3)            list [1, 3]
  addAtIndex(1, 2)            list [1, 2, 3]
  get(1) -> 2      list [1, 2, 3]
  deleteAtIndex(

## After it passes

- **Count your `if`s.** Write `addAtIndex` and `deleteAtIndex` once *without* the
  sentinel, then with it, and count the branches in each version. That difference
  is the whole reason sentinels exist, and it is the answer you give when someone
  asks why you added a node that holds no data.
- **Build route B and re-run the same tests.** Then add the optimisation the
  doubly linked version makes possible: `get(i)` walks from the *nearer* end.
  Measure it - `timeit` `get(1999)` on a 2000-element list, before and after. Is
  the factor what you predicted?
- **Fill in question 7's table with real numbers.** Time `list.insert(0, x)` and
  your `addAtHead` for `n = 10 000`, and `list[i]` against your `get(i)`. Then
  write the two-sentence version of "when is a linked list the right answer?"
  Hint: the honest answer mentions *holding a pointer to a node you already have*
  - which is exactly what an index-based API like this one never lets you do.
- **Then look at what you have really built.** A doubly linked list with `O(1)`
  splice-out and splice-in, plus a `dict` from key to node, **is** an LRU cache
  (#146). You are one dict away. That is the reason 707 is worth doing properly
  rather than passing.
- **Add `reverse()` to your class** - #206, but now on a list you own, with a
  `size` and a `tail` to keep consistent. Which of your invariants does reversing
  break if you only rewrite `next` pointers?
- **The invariant list.** Write down every fact your class promises itself
  (`size` equals the number of real nodes; `tail.prev` is the last real node;
  every `a.next.prev is a`; ...) and then, for each method, which ones it could
  break. That list is what a `pytest` file would assert after every operation -
  which is what `check` does above, and why it finds bugs at the call that caused
  them instead of three calls later.
- Siblings: **#146 LRU Cache** (this class + a dict, and the interview favourite),
  #1472 Design Browser History (a doubly linked list wearing a costume - or two
  stacks, and comparing the two is the fun part), #641 Design Circular Deque
  (#622 with four methods), #206 Reverse Linked List (you already did it; now do
  it *inside* a class that has to stay consistent).
